# ARC-3 serve-verify K3 — merge ladder checkpoint into Qwen3.6-27B and prove it serves

July's fine-tune died because a LoRA never reached the served model. This gate proves, in one
offline RTX-6000 commit, that a ladder checkpoint (default `checkpoint-16`, the OOD-early-peak arm):
1. **merges** into the base with exactly the trainer's load semantics (delta == B@A x alpha/r, checked),
2. **serves** under the duck's exact vLLM line (bf16; FP8 requant deferred to the publish step),
3. **behaves**: greedy val-prompt probes emit well-formed `<tool_call>` python calls,
4. differs from the BASE FP8 snapshot on the same probes (reported; precision-confounded).
Output = logs + `gate_result.json` only; the merged model stays in scratch by design (20GB cap).


In [ ]:
import glob, json, os, shutil, signal, subprocess, sys, time, urllib.request
deps = sorted(glob.glob("/kaggle/input/**/deps", recursive=True))
if deps: sys.path.insert(0, deps[0])
print("deps on path:", deps[:1])
import torch
DEV = torch.cuda.get_device_name(0) if torch.cuda.is_available() else "CPU"
RUN = "RTX PRO 6000" in DEV.upper() or os.environ.get("GATE_FORCE") == "1"
print(f"device: {DEV} | RUN={RUN}")
import transformers, peft
print("transformers", transformers.__version__, "| peft", peft.__version__)
GATE_CKPT = os.environ.get("GATE_CKPT", "checkpoint-8")  # run-8 ladder: checkpoint-8 or sft_adapter (step-15 final)
GATE_MAX_TOKENS = int(os.environ.get("GATE_MAX_TOKENS", 6144))  # thinking is served-on; leave room before the tool_call


In [ ]:
GATE_MERGE_SRC = '# gate_merge.py — merge + NLL as a SUBPROCESS.\n# In-process, del/gc.collect()/empty_cache() still left 59,475 MiB resident\n# (measured, serve-verify v7), so vLLM died at startup:\n#   "Free memory on device cuda:0 (36.35/94.97 GiB) ... less than desired GPU\n#    memory utilization (0.9, 85.47 GiB)".\n# Some reference survives in the peft/transformers graph. Rather than hunt it, do\n# what the duck-sft submission already does and let the OS reclaim every byte at\n# process exit — which also makes this gate mirror the submission\'s real path.\nimport glob, json, os, random, shutil, subprocess, sys, time\n\n# A subprocess does NOT inherit the notebook\'s sys.path. The vendored deps\n# (arc3-deps-prep) carry transformers 5.14.0.dev0, which is the only build that\n# knows model_type \'qwen3_5\'; the system transformers raises\n#   ValueError: checkpoint has model type `qwen3_5` but Transformers does not\n#   recognize this architecture\n# So replicate the notebook guard cell\'s deps insert BEFORE importing torch or\n# transformers, and fail loudly if the deps tree is absent rather than silently\n# falling back to a build that cannot load the model.\n_deps = sorted(glob.glob("/kaggle/input/**/deps", recursive=True))\nassert _deps, "arc3-deps-prep deps/ not found — refusing to run on system transformers"\nsys.path.insert(0, _deps[0])\nprint(f"[gate_merge] deps on path: {_deps[0]}", flush=True)\n\nimport torch\nimport transformers\nprint(f"[gate_merge] transformers {transformers.__version__}", flush=True)\n\n_IN = json.load(open("/kaggle/working/gate_inputs.json"))\nMODEL, CKPT = _IN["MODEL"], _IN["CKPT"]\nCORPUS, MERGED, GATE_CKPT = _IN["CORPUS"], _IN["MERGED"], _IN["GATE_CKPT"]\nos.environ.setdefault("GATE_NLL_MAX_LEN", str(_IN["NLL_MAX_LEN"]))\nsys.path.insert(0, CORPUS)\nfrom sft_common import dequantize_fp8_inplace, strip_quantization_runtime\nTOOLS = json.load(open(os.path.join(CORPUS, "tools.json")))\nval_rows = [json.loads(l) for l in open(os.path.join(CORPUS, "val.jsonl"))]\n\nfrom transformers import AutoModelForImageTextToText, AutoProcessor\nt0 = time.time()\nmodel = AutoModelForImageTextToText.from_pretrained(\n    MODEL, torch_dtype=torch.bfloat16, device_map={"": 0})\nfor a in ("quantization_config", "_pre_quantization_dtype"):\n    if hasattr(model.config, a):\n        try: setattr(model.config, a, None)\n        except Exception: pass\nmodel.is_quantized = False\nif hasattr(model, "hf_quantizer"): model.hf_quantizer = None\n# 07-26 fix: apply FP8 scales BEFORE strip deletes them (v1 died in merge on f8 +=;\n# same root cause invalidated training runs 1-5)\nprint("dequant:", dequantize_fp8_inplace(model))\nprint("strip:", strip_quantization_runtime(model), f"| load {time.time()-t0:.0f}s")\ndt = {str(p.dtype) for p in model.parameters()}\nassert "torch.float8_e4m3fn" not in dt, dt\n\n# cache 3 targeted base weights for the post-merge delta assert\nacfg = json.load(open(os.path.join(CKPT, "adapter_config.json")))\nscaling = acfg["lora_alpha"] / acfg["r"]\nfrom peft import PeftModel\npmodel = PeftModel.from_pretrained(model, CKPT, is_trainable=False)\nlora_mods = [(n, m) for n, m in pmodel.named_modules()\n             if hasattr(m, "lora_A") and "default" in getattr(m, "lora_A", {})]\nassert len(lora_mods) > 400, f"adapter did not attach: {len(lora_mods)} lora modules"\nimport random; random.seed(0)\nchecks = []\nfor n, m in random.sample(lora_mods, 3):\n    BA = (m.lora_B["default"].weight @ m.lora_A["default"].weight) * scaling\n    checks.append((n, m.base_layer.weight.detach().clone(), BA.detach().clone()))\n# ---- NLL: adapter-attached vs base vs merged -------------------------------\n# The weight-level rel_err below is a proxy; THIS is the quantity that matters.\n# 2026-07-31: the merge assert failed with rel_err 0.64-0.75 across modules\n# spanning a 20x range of delta magnitudes. A synthetic reproduction showed\n# that is exactly what bf16 STORAGE of a delta ~1.5e-3 the size of the weights\n# produces (bf16 0.696 / fp16 0.138 / fp32 0.000) — and that accumulating the\n# merge in fp32 does NOT help, because the final cast is what destroys it.\n# So measure whether the merged model actually keeps the fine-tune\'s NLL gain,\n# rather than inferring model quality from weight fidelity.\nfrom sft_common import encode_with_mask\n_nll_rows = val_rows[:4]\n# Memory history, both real bugs found by running this gate:\n#  v3: full forward at ctx 32768 OOM\'d — HF materializes logits for EVERY position,\n#      ~33k x ~152k vocab x 2B ~ 10GB (x2 more when upcast to fp32).\n#  v4: capping ctx at 4096 made encode_with_mask drop ALL rows as untruncatable\n#      (val rows average ~21k tokens and carry images), so every NLL came back\n#      0.0 over 0 tokens and the gate then "diagnosed" a dead checkpoint from an\n#      EMPTY measurement. A metric that cannot detect its own invalidity is worse\n#      than no metric — hence the hard ntok assert below.\n# Correct fix: keep the full context, but only compute logits for the scored\n# suffix. The loss mask is the target only (~1.5k tokens), so logits_to_keep\n# shrinks the hog by ~14x without touching what is measured.\nNLL_MAX_LEN = int(os.environ.get("GATE_NLL_MAX_LEN", 32768))\n\ndef _row_nll(mdl, feats):\n    ids = feats["input_ids"][0]\n    n_tgt = int((feats["labels"][0] != -100).sum())\n    fwd = {k: v for k, v in feats.items() if k != "labels"}\n    with torch.no_grad():\n        try:\n            out = mdl(**fwd, logits_to_keep=n_tgt + 1)\n        except TypeError:  # older signature\n            out = mdl(**fwd, num_logits_to_keep=n_tgt + 1)\n        lg = out.logits[0, :-1].float()          # predicts the last n_tgt tokens\n        tgt = ids[-n_tgt:].to(lg.device)\n        loss = torch.nn.functional.cross_entropy(lg, tgt, reduction="mean")\n    return float(loss), n_tgt\n\ndef _nll(mdl, tag):\n    tot, ntok, used, skipped = 0.0, 0, 0, []\n    for r in _nll_rows:\n        feats, info = encode_with_mask(proc_for_nll, r["messages"], r["target"],\n                                       TOOLS, NLL_MAX_LEN)\n        if feats is None:\n            skipped.append(info.get("error"))\n            continue\n        feats = {k: (v.to(0) if hasattr(v, "to") else v) for k, v in feats.items()}\n        v, n = _row_nll(mdl, feats)\n        tot += v * n\n        ntok += n\n        used += 1\n        del feats\n        torch.cuda.empty_cache()\n    # NEVER return a number derived from zero rows — that is what made v4 lie.\n    assert ntok > 0, (\n        f"NLL probe measured NOTHING for {tag}: 0 of {len(_nll_rows)} rows encoded "\n        f"(errors={skipped}, ctx<={NLL_MAX_LEN}). Fix the probe before reading any "\n        f"verdict — a zero here is a broken instrument, not a dead adapter.")\n    v = tot / ntok\n    print(f"[nll] {tag}: {v:.4f} over {ntok} target tokens "\n          f"({used}/{len(_nll_rows)} rows, ctx<={NLL_MAX_LEN})", flush=True)\n    return v\n\nfrom transformers import AutoProcessor as _AP\ntry:\n    proc_for_nll = _AP.from_pretrained(MODEL)\nexcept Exception:\n    proc_for_nll = _AP.from_pretrained(os.path.join(CORPUS, "tokenizer_bundle"))\n\nnll_adapter = _nll(pmodel, "adapter attached (unmerged)")\nwith pmodel.disable_adapter():\n    nll_base = _nll(pmodel, "base (adapter disabled)")\n\nmerged = pmodel.merge_and_unload()\nnll_merged = _nll(merged, "merged")\n\ngain_attached = nll_base - nll_adapter\ngain_merged = nll_base - nll_merged\nretained = gain_merged / gain_attached if abs(gain_attached) > 1e-6 else 0.0\nprint(f"[nll] base={nll_base:.4f} attached={nll_adapter:.4f} merged={nll_merged:.4f}")\nprint(f"[nll] gain attached={gain_attached:+.4f} merged={gain_merged:+.4f} "\n      f"RETAINED={retained:.1%}", flush=True)\nresults_nll = {"base": nll_base, "attached": nll_adapter, "merged": nll_merged,\n               "gain_attached": gain_attached, "gain_merged": gain_merged,\n               "retained_fraction": retained}\n# ----------------------------------------------------------------------------\n\nok = []\n_named = dict(merged.named_modules())\ndef _resolve(name):\n    # peft\'s merge_and_unload strips the \'base_model.model.\' prefix that\n    # pre-merge names carry (KeyError on raw lookup, peft 0.19.x) —\n    # audit fix 2026-07-26, mirrored in serving_assert._resolve_merged_module\n    for c in (name, name.removeprefix("base_model.model."), "base_model.model." + name):\n        if c in _named: return _named[c]\n    raise KeyError(f"module {name!r} not found post-merge (tried prefix variants)")\nfor (n, w_pre, BA) in checks:\n    w_post = _resolve(n).weight.detach()\n    delta = (w_post - w_pre).float(); exp = BA.float()\n    rel = (delta - exp).norm() / (exp.norm() + 1e-9)\n    ok.append({"module": n, "delta_norm": float(exp.norm()), "rel_err": float(rel)})\n    print(f"[delta] {n}: |BA|={exp.norm():.4f} rel_err={rel:.4f}")\n# The weight-level delta is now DIAGNOSTIC, not pass/fail. A bf16 merge of a\n# delta this small is provably lossy (see the note above), so a high rel_err\n# here is expected and is not by itself evidence the adapter is broken — the\n# NLL block already measures what actually matters. Kept because delta_norm==0\n# would still mean the adapter never attached, which IS fatal.\njson.dump({"nll": results_nll, "deltas": ok},\n          open("/kaggle/working/gate_merge_result.json", "w"), indent=2)\nprint("[gate] wrote gate_merge_result.json", flush=True)\nassert all(c["delta_norm"] > 0 for c in ok), f"adapter contributed nothing: {ok}"\nif any(c["rel_err"] >= 0.05 for c in ok):\n    print(f"[warn] weight deltas degraded by merge rounding (expected for bf16): {ok}",\n          flush=True)\n\n# A1-PROTOCOL §1 pre-registers: merged target-NLL gain >= 2% on the val rows.\nassert gain_attached > 0, (\n    f"adapter gives NO NLL gain even attached — the checkpoint, not the merge, "\n    f"is the problem: {results_nll}")\nrel_gain_merged = gain_merged / nll_base if nll_base else 0.0\nprint(f"[nll] merged relative gain = {rel_gain_merged:.2%} (A1 §1 requires >= 2%)")\nassert rel_gain_merged >= 0.02, (\n    f"MERGE GATE FAIL — merged model does not carry the fine-tune "\n    f"({rel_gain_merged:.2%} < 2%; {retained:.1%} of the attached gain survived). "\n    f"Serve the adapter unmerged via vLLM --enable-lora instead. {results_nll}")\nprint("MERGE GATE: PASS")\n\nmerged.config.torch_dtype = torch.bfloat16\nmerged.config.use_cache = True\n# merge_and_unload() hands back a model that still carries the compressed-tensors\n# plumbing (the "Compressing/Decompressing model" bars in the log). save_pretrained\n# then routes through a quantizer whose compressor is None ->\n# AttributeError: \'NoneType\' object has no attribute \'convert\'. The weights are\n# already true bf16 at this point (dequant applied, 0 f8 params), so the right move\n# is to detach the quantization path entirely before writing.\nfor _a in ("hf_quantizer", "_hf_peft_config_loaded"):\n    if hasattr(merged, _a):\n        try: setattr(merged, _a, None)\n        except Exception: pass\nmerged.is_quantized = False\nfor _a in ("quantization_config", "_pre_quantization_dtype", "compression_config"):\n    if hasattr(merged.config, _a):\n        try: delattr(merged.config, _a)\n        except Exception:\n            try: setattr(merged.config, _a, None)\n            except Exception: pass\n_left = [n for n, p in merged.named_parameters() if p.dtype == torch.float8_e4m3fn]\nassert not _left, f"f8 params present at save time: {_left[:5]}"\nprint(f"[save] quantization detached; dtypes={sorted({str(p.dtype) for p in merged.parameters()})}",\n      flush=True)\nt0 = time.time()\n# save_original_format defaults True in transformers 5.14.0.dev0, which runs\n# revert_weight_conversion() -> mapping.convert() where one op is None:\n#   AttributeError: \'NoneType\' object has no attribute \'convert\'\n# We are feeding vLLM, which reads HF format, so reverting to the original\n# checkpoint layout is both broken here and not what we want.\ntry:\n    merged.save_pretrained(MERGED, safe_serialization=True, max_shard_size="4GB",\n                           save_original_format=False)\nexcept TypeError:  # older transformers without the kwarg\n    merged.save_pretrained(MERGED, safe_serialization=True, max_shard_size="4GB")\ntry:\n    proc = AutoProcessor.from_pretrained(MODEL)\nexcept Exception as e:\n    print("processor from snapshot failed:", e)\n    proc = AutoProcessor.from_pretrained(os.path.join(CORPUS, "tokenizer_bundle"))\nproc.save_pretrained(MERGED)\nfor extra in ("chat_template.jinja", "chat_template.json"):\n    src = os.path.join(MODEL, extra)\n    if os.path.exists(src) and not os.path.exists(os.path.join(MERGED, extra)):\n        shutil.copy(src, MERGED)\ncfg = json.load(open(os.path.join(MERGED, "config.json")))\nfor k in ("quantization_config", "_pre_quantization_dtype", "compression_config"):\n    cfg.pop(k, None)\njson.dump(cfg, open(os.path.join(MERGED, "config.json"), "w"), indent=2)\nprint(f"saved {MERGED} in {time.time()-t0:.0f}s:", sorted(os.listdir(MERGED))[:8], "...")\ndel merged, pmodel, model, checks\nimport gc; gc.collect(); torch.cuda.empty_cache()\nprint(subprocess.run(["nvidia-smi", "--query-gpu=memory.used", "--format=csv"],\n                     capture_output=True, text=True).stdout)\n'
CORPUS = os.path.dirname(sorted(glob.glob("/kaggle/input/**/train.jsonl", recursive=True))[0])
sys.path.insert(0, CORPUS)
from sft_common import dequantize_fp8_inplace, strip_quantization_runtime
MODEL = next(os.path.dirname(p) for p in glob.glob("/kaggle/input/**/config.json", recursive=True)
             if "tokenizer_bundle" not in p and json.load(open(p)).get("model_type") == "qwen3_5")
# run-8 artifacts live in TWO shapes: sft_out/checkpoint-N (mid-run saves) and
# sft_adapter/ (the step-15 final, NOT under sft_out/) — glob must cover both.
_ckpt_candidates = sorted(glob.glob("/kaggle/input/**/sft_out/checkpoint-*", recursive=True))                  + sorted(os.path.dirname(p) for p in glob.glob("/kaggle/input/**/sft_adapter/adapter_config.json", recursive=True))
CKPT = next(p for p in _ckpt_candidates if p.rstrip("/").endswith(GATE_CKPT))
WHEELHOUSE = os.path.dirname(sorted(glob.glob("/kaggle/input/**/requirements.lock", recursive=True))[0])
print("model:", MODEL, "\nckpt:", CKPT, "\nwheelhouse:", WHEELHOUSE, "\ncorpus:", CORPUS)
TOOLS = json.load(open(os.path.join(CORPUS, "tools.json")))
val_rows = [json.loads(l) for l in open(os.path.join(CORPUS, "val.jsonl"))]
# fixed probes: first val row WITH an image, first val row WITHOUT, chosen deterministically
def has_image(r):
    return any(isinstance(m.get("content"), list) and any(p.get("type") == "image_url" for p in m["content"])
               for m in r["messages"])
PROBE_IMG = next(r for r in val_rows if has_image(r))
# corpus is fully multimodal (all 43 val rows carry images) -> probe B is just a distinct second row,
# text-only if one ever exists
PROBE_TXT = next((r for r in val_rows if not has_image(r)),
                 next(r for r in val_rows if r is not PROBE_IMG))
print("probe A (image):", len(PROBE_IMG["messages"]), "msgs | probe B:", len(PROBE_TXT["messages"]), "msgs")
MERGED = "/tmp/merged_" + GATE_CKPT


In [ ]:
if RUN:
    # The merge + NLL run in a SUBPROCESS so the OS reclaims 100% of VRAM before
    # vLLM starts. In-process cleanup measurably did NOT free the 27B (59,475 MiB
    # still held after del/gc/empty_cache in v7), and vLLM then refused to start.
    # duck-sft already merges this way; the gate now mirrors it.
    json.dump({"MODEL": MODEL, "CKPT": CKPT, "CORPUS": CORPUS,
               "MERGED": MERGED, "GATE_CKPT": GATE_CKPT,
               "NLL_MAX_LEN": int(os.environ.get("GATE_NLL_MAX_LEN", 32768))},
              open("/kaggle/working/gate_inputs.json", "w"))
    _script_path = "/kaggle/working/gate_merge.py"
    open(_script_path, "w").write(GATE_MERGE_SRC)
    _t0 = time.time()
    _rc = subprocess.run([sys.executable, _script_path]).returncode
    print(f"[gate] merge subprocess rc={_rc} in {time.time()-_t0:.0f}s", flush=True)
    _free = subprocess.run(["nvidia-smi", "--query-gpu=memory.used",
                            "--format=csv,noheader"],
                           capture_output=True, text=True).stdout.strip()
    print(f"[gate] VRAM used after subprocess exit: {_free}", flush=True)
    _rp = "/kaggle/working/gate_merge_result.json"
    merge_result = json.load(open(_rp)) if os.path.exists(_rp) else None
    if merge_result:
        print("[gate] nll:", json.dumps(merge_result["nll"]), flush=True)
    assert _rc == 0, f"merge subprocess FAILED rc={_rc} (result={merge_result})"
    assert os.path.exists(os.path.join(MERGED, "config.json")),         f"merged tree missing at {MERGED}"
    print("MERGE STAGE OK — proceeding to vLLM serve check", flush=True)


In [ ]:
if RUN:
    SITE = "/kaggle/working/vllm-site-packages"
    if not os.path.exists(os.path.join(SITE, "vllm")):
        subprocess.run([sys.executable, "-m", "pip", "install", "--no-index",
                        "--find-links", WHEELHOUSE, "--requirement",
                        os.path.join(WHEELHOUSE, "requirements.lock"), "--target", SITE,
                        "--upgrade", "--ignore-installed", "--only-binary", ":all:",
                        "--no-compile", "--disable-pip-version-check", "--no-warn-conflicts"],
                       check=True, capture_output=True)
    ENV = dict(os.environ, PYTHONPATH=SITE, USE_TF="0", TRANSFORMERS_NO_TF="1",
               TRANSFORMERS_NO_TORCHVISION="1", VLLM_NO_USAGE_STATS="1")
    v = subprocess.run([sys.executable, "-c", "import vllm; print(vllm.__version__)"],
                       env=ENV, capture_output=True, text=True)
    print("vllm:", v.stdout.strip(), v.stderr.strip()[-200:])
    assert v.returncode == 0, "vLLM import failed from wheelhouse site-packages"


In [ ]:
BASE_URL = "http://127.0.0.1:1234/v1"

def req(url, payload=None, timeout=30):
    data = None if payload is None else json.dumps(payload).encode()
    r = urllib.request.Request(url, data=data, headers={"Content-Type": "application/json"})
    with urllib.request.urlopen(r, timeout=timeout) as resp:
        return json.loads(resp.read().decode())

def start_server(model_path, log_path):
    # the duck's EXACT serve line (setup_commands.json), model path swapped
    cmd = [sys.executable, "-m", "vllm.entrypoints.openai.api_server",
           "--model", model_path, "--served-model-name", "gate/model",
           "--host", "127.0.0.1", "--port", "1234", "--tensor-parallel-size", "1",
           "--enable-auto-tool-choice", "--tool-call-parser", "qwen3_coder",
           "--generation-config", "vllm", "--enable-prefix-caching",
           "--default-chat-template-kwargs", '{"preserve_thinking": true}',
           "--reasoning-parser", "qwen3", "--max-model-len", "65536"]
    lh = open(log_path, "w")
    p = subprocess.Popen(cmd, env=ENV, stdout=lh, stderr=subprocess.STDOUT, text=True)
    deadline = time.monotonic() + 1800
    while time.monotonic() < deadline:
        if p.poll() is not None:
            print(open(log_path).read()[-4000:])
            raise RuntimeError(f"vLLM died rc={p.returncode} for {model_path}")
        try:
            req(BASE_URL + "/models", timeout=5); print("server ready:", model_path); return p
        except Exception:
            time.sleep(5)
    print(open(log_path).read()[-4000:])
    raise TimeoutError("vLLM never became ready")

def stop_server(p):
    p.send_signal(signal.SIGTERM)
    try: p.wait(60)
    except subprocess.TimeoutExpired: p.kill(); p.wait(30)
    for _ in range(36):
        used = subprocess.run(["nvidia-smi", "--query-gpu=memory.used", "--format=csv,noheader,nounits"],
                              capture_output=True, text=True).stdout.strip()
        if used and int(used.split()[0]) < 8000: break
        time.sleep(5)
    print("gpu after stop:", used, "MiB")

def _probe_messages(row):
    # Corpus rows carry assistant turns with tool_calls of type "function", which this
    # vLLM chat parser rejects outright:
    #   pydantic ValidationError ... Input should be 'custom' [input_value='function']
    # -> HTTP 400 before the model ever generates (observed, serve-verify v9). That is a
    # request-schema mismatch in the PROBE, not a property of the merged model: the duck
    # harness builds its own payloads instead of replaying corpus rows. Strip tool_calls
    # from prior assistant turns so the probe reaches generation. Both arms get identical
    # treatment, so the merged-vs-base comparison stays paired.
    out = []
    for m in row["messages"]:
        if m.get("role") == "assistant" and m.get("tool_calls"):
            m = {k: v for k, v in m.items() if k != "tool_calls"}
            if not m.get("content"):
                m["content"] = ""
        out.append(m)
    return out


def probe(tag):
    out = {}
    for name, row in (("val_img", PROBE_IMG), ("val_txt", PROBE_TXT)):
        r = req(BASE_URL + "/chat/completions",
                {"model": "gate/model", "messages": _probe_messages(row), "tools": TOOLS,
                 "temperature": 0.0, "max_tokens": GATE_MAX_TOKENS}, timeout=900)
        msg = r["choices"][0]["message"]
        tcs = msg.get("tool_calls") or []
        wf = bool(tcs) and all(isinstance(json.loads(t["function"]["arguments"])
                                          if isinstance(t["function"]["arguments"], str)
                                          else t["function"]["arguments"], dict) for t in tcs)
        out[name] = {"tool_calls": [t["function"]["name"] for t in tcs], "well_formed": wf,
                     "content": (msg.get("content") or "")[:400],
                     "reasoning": (msg.get("reasoning_content") or "")[:200],
                     "raw_args": [str(t["function"]["arguments"])[:300] for t in tcs]}
        print(f"[{tag}:{name}] tools={out[name]['tool_calls']} well_formed={wf}")
    r = req(BASE_URL + "/chat/completions",
            {"model": "gate/model", "temperature": 0.0, "max_tokens": 128,
             "chat_template_kwargs": {"enable_thinking": False},
             "messages": [{"role": "user", "content": "Answer in one short sentence: what is 17 * 23?"}]},
            timeout=180)
    out["plain"] = {"content": r["choices"][0]["message"].get("content", "").strip()}
    print(f"[{tag}:plain] {out['plain']['content'][:120]}")
    return out


In [ ]:
if RUN:
    results = {"ckpt": GATE_CKPT}
    p = start_server(MERGED, "/kaggle/working/vllm-merged.log")
    results["merged"] = probe("merged")
    stop_server(p)
    p = start_server(MODEL, "/kaggle/working/vllm-base.log")
    results["base"] = probe("base")
    stop_server(p)

    differ = {k: results["merged"][k] != results["base"][k] for k in ("val_img", "val_txt", "plain")}
    results["outputs_differ"] = differ
    merged_wf = all(results["merged"][k]["well_formed"] for k in ("val_img", "val_txt"))
    results["verdict"] = {"merged_serves": True, "merged_tool_calls_well_formed": merged_wf,
                          "differs_from_base_anywhere": any(differ.values())}
    json.dump(results, open("/kaggle/working/gate_result.json", "w"), indent=2)
    print("\n" + "=" * 80)
    print(f"GATE VERDICT ({GATE_CKPT}): serves=True well_formed={merged_wf} differ={differ}")
    print("=" * 80)
    assert merged_wf, "merged model did not emit well-formed tool_calls on the val probes"
    print("SERVING-VERIFICATION GATE: PASS")
